# Structural variation and dynamic loop extrusion

This tutorial carries a synthetic inversion from a locus interaction matrix into a seeded loop-extrusion trajectory and a short OpenMiChroM simulation. Dynamic harmonic bonds are updated inside one OpenMM Context, so positions and velocities remain in memory and are not rounded through an intermediate coordinate file.

All locus and particle indices are zero-based. A trajectory with n transitions contains n + 1 frames because it includes the initial state.


The default fast mode is a smoke test suitable for an ordinary laptop. Set OPENMICHROM_TUTORIAL_MODE=full before starting Jupyter for a longer demonstration. The example parameters are synthetic and test software mechanics; they are not a fitted biological model.


In [ ]:
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from OpenMiChroM.ChromDynamics import MiChroM
from OpenMiChroM.Extrusion_Bonds import LoopExtrusionManager
from OpenMiChroM.StructuralVariants import (
    apply_structural_variant,
    locus_labels,
    write_locus_matrix,
    write_locus_sequence,
)

TUTORIAL_MODE = os.environ.get("OPENMICHROM_TUTORIAL_MODE", "fast").lower()
TUTORIAL_FAST = TUTORIAL_MODE != "full"
TUTORIAL_PLATFORM = os.environ.get("OPENMICHROM_TUTORIAL_PLATFORM", "CPU")
transition_count = 8 if TUTORIAL_FAST else 40
md_steps_per_frame = 2 if TUTORIAL_FAST else 20
print(
    f"mode={'fast' if TUTORIAL_FAST else 'full'}, platform={TUTORIAL_PLATFORM}, "
    f"transitions={transition_count}, MD steps/frame={md_steps_per_frame}"
)


In [ ]:
size = 24
loci = np.arange(size)
separation = np.abs(loci[:, None] - loci[None, :])
base_matrix = (
    -0.24 * np.exp(-separation / 4.0)
    - 0.015 * np.cos((loci[:, None] + loci[None, :]) / 3.0)
)
base_matrix = 0.5 * (base_matrix + base_matrix.T)

forward = np.full(size, 0.02)
reverse = np.full(size, 0.02)
forward[[5, 12, 18]] = [0.70, 0.85, 0.65]
reverse[[7, 14, 20]] = [0.60, 0.75, 0.80]

variant = apply_structural_variant(
    base_matrix,
    "inversion",
    8,
    16,
    forward_motifs=forward,
    reverse_motifs=reverse,
    adjust_ideal_chromosome=True,
)
assert variant.matrix.shape == base_matrix.shape
print("Variant index map:", variant.index_map.tolist())


In [ ]:
manager = LoopExtrusionManager(
    variant.forward_motifs,
    variant.reverse_motifs,
    num_steps=transition_count,
    extruder_count=3,
    off_rate=0.08,
    unfix_conversion_factor=1 / 4000,
    seed=2026,
)
trajectory = manager.simulate()

assert trajectory.shape == (transition_count + 1, 3, 2)
assert trajectory.min() >= 0
assert trajectory.max() < size
for frame in trajectory:
    anchors = frame.ravel()
    assert np.unique(anchors).size == anchors.size
    assert np.all(frame[:, 0] < frame[:, 1])

same_seed = LoopExtrusionManager(
    variant.forward_motifs,
    variant.reverse_motifs,
    num_steps=transition_count,
    extruder_count=3,
    off_rate=0.08,
    seed=2026,
).simulate()
np.testing.assert_array_equal(trajectory, same_seed)
print(f"Generated and checked {trajectory.shape[0]} deterministic bond frames.")


In [ ]:
fig, axis = plt.subplots(figsize=(8, 3.8), constrained_layout=True)
steps = np.arange(trajectory.shape[0])
for motor in range(trajectory.shape[1]):
    axis.plot(steps, trajectory[:, motor, 0], marker="o", ms=3, label=f"motor {motor} left")
    axis.plot(steps, trajectory[:, motor, 1], marker="o", ms=3, linestyle="--", label=f"motor {motor} right")
axis.set(xlabel="extrusion frame", ylabel="zero-based bead index", title="Seeded loop-anchor trajectory")
axis.legend(ncol=3, fontsize=8)
plt.show()


## Run one persistent OpenMM Context

The loop force contains the union of every particle pair used by the trajectory. Switching frames changes only an active per-bond parameter and calls OpenMM parameter updating; it does not rebuild the System or Context.


In [ ]:
temporary = tempfile.TemporaryDirectory(prefix="openmichrom-extrusion-")
output_dir = Path(temporary.name)
labels = locus_labels(size)
matrix_path = write_locus_matrix(
    output_dir / "variant_lambdas.csv",
    variant.matrix,
    labels=labels,
)
sequence_path = write_locus_sequence(
    output_dir / "variant_sequence.txt",
    labels,
)
print(f"Prepared simulation inputs in {output_dir}")


In [ ]:
simulation = MiChroM(name="SVLoopExtrusion", verbose=False)
simulation.setup(platform=TUTORIAL_PLATFORM, printing=False)
if hasattr(simulation.integrator, "setRandomNumberSeed"):
    simulation.integrator.setRandomNumberSeed(2026)
simulation.saveFolder(str(output_dir))

positions = simulation.initStructure(
    mode="spring",
    ChromSeq=str(sequence_path),
    isRing=False,
)
simulation.loadStructure(positions)
simulation.addFENEBonds()
simulation.addAngles(kA=2.0)
simulation.addRepulsiveSoftCore(eCut=4.0)
simulation.addCustomTypes(TypesTable=str(matrix_path))
simulation.addDynamicLoopPotential(
    trajectory,
    k_loop=8.0,
    r0_loop=1.0,
)
simulation.createSimulation(printing=False)

position_frames = []
for frame_index in range(trajectory.shape[0]):
    if frame_index:
        simulation.updateDynamicLoopPotential(frame_index)
    simulation.run(md_steps_per_frame, report=False)
    current = simulation.getPositions().copy()
    assert np.isfinite(current).all()
    assert np.isfinite(simulation.getVelocities()).all()
    position_frames.append(current)

position_frames = np.asarray(position_frames)
assert simulation.contexted
print(
    f"Completed {position_frames.shape[0] * md_steps_per_frame} MD steps "
    "without rebuilding the context."
)


In [ ]:
contact_sum = np.zeros((size, size), dtype=float)
for coordinates in position_frames:
    distances = np.linalg.norm(coordinates[:, None, :] - coordinates[None, :, :], axis=-1)
    contact_sum += distances < 1.78
contact_probability = contact_sum / len(position_frames)
np.fill_diagonal(contact_probability, 0.0)

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.5), constrained_layout=True)
axes[0].imshow(variant.matrix, cmap="coolwarm")
axes[0].set_title("variant interaction matrix")
image = axes[1].imshow(contact_probability, vmin=0, vmax=1, cmap="magma")
axes[1].set_title("short-run contact preview")
for axis in axes:
    axis.set(xlabel="locus", ylabel="locus")
fig.colorbar(image, ax=axes[1], label="contact fraction")
plt.show()

print(
    f"Contact preview range: {contact_probability.min():.3f} to "
    f"{contact_probability.max():.3f}"
)


In [ ]:
del simulation.simulation
del simulation.context
temporary.cleanup()
print("OpenMM context released and temporary tutorial output removed.")


## Interpretation and production use

The short contact map above is only a runtime check. Production work needs calibrated motif probabilities, an experimentally justified interaction matrix, equilibration, convergence checks, multiple independent seeds, and uncertainty analysis. Record the random seed and save the full extrusion trajectory with the simulation metadata.

This maintained workflow grew from [OpenMiChroM pull request 123](https://github.com/junioreif/OpenMiChroM/pull/123) by Miles Gantcher. The extrusion concept is related to [Sanborn et al., PNAS 2015](https://doi.org/10.1073/pnas.1518552112). Biological structural-variant context is discussed by [Lupiáñez et al., Cell 2015](https://doi.org/10.1016/j.cell.2015.04.004), [Bianco et al., Nature Genetics 2018](https://doi.org/10.1038/s41588-018-0098-8), and [Andrey et al., Genome Research 2017](https://doi.org/10.1101/gr.213066.116).
